In [ ]:
from pynq import Overlay, allocate
import numpy as np
import time

overlay = Overlay('layer1_stream.bit')
dma = overlay.axi_dma
dma_send = overlay.axi_dma.sendchannel
dma_recv = overlay.axi_dma.recvchannel
hls_ip = overlay.layer1_stream_0

print(overlay.ip_dict.keys())

dict_keys(['layer1_stream_0', 'axi_dma', 'zynq_ultra_ps_e_0'])


In [ ]:
print(hls_ip.register_map)

RegisterMap {
  CTRL = Register(AP_START=0, AP_DONE=0, AP_IDLE=1, AP_READY=0, RESERVED_1=0, AUTO_RESTART=0, RESERVED_2=0, INTERRUPT=0, RESERVED_3=0),
  GIER = Register(Enable=0, RESERVED=0),
  IP_IER = Register(CHAN0_INT_EN=0, CHAN1_INT_EN=0, RESERVED_0=0),
  IP_ISR = Register(CHAN0_INT_ST=0, CHAN1_INT_ST=0, RESERVED_0=0)
}


In [ ]:
INPUT_SIZE  = 784
OUTPUT_SIZE = 128
W_SIZE      = INPUT_SIZE * OUTPUT_SIZE  # 100352
TOTAL_IN    = INPUT_SIZE + W_SIZE + OUTPUT_SIZE  # 101264
TOTAL_OUT   = OUTPUT_SIZE
SCALE       = 16

print(f'Input buffer: {TOTAL_IN} values = {TOTAL_IN * 4} bytes')
print(f'Output buffer: {TOTAL_OUT} values = {TOTAL_OUT * 4} bytes')

Input buffer: 101264 values = 405056 bytes
Output buffer: 128 values = 512 bytes


In [ ]:
W1 = np.load('w1.npy')  # (784, 128) float32
b1 = np.load('b1.npy')  # (128,) float32
W2 = np.load('w2.npy')  # (128, 10) float32
b2 = np.load('b2.npy')  # (10,) float32

W1_int = (W1.T * SCALE).astype(np.int32)        # (128, 784)
b1_int = (b1 * SCALE * SCALE).astype(np.int32)  # (128,)

print('W1:', W1.shape, 'W1_int:', W1_int.shape)
print('W1_int range:', W1_int.min(), 'to', W1_int.max())
print('b1_int range:', b1_int.min(), 'to', b1_int.max())

W1: (784, 128) W1_int: (128, 784)
W1_int range: -12 to 8
b1_int range: -46 to 47


In [ ]:
def load_idx_images(path):
    with open(path, 'rb') as f:
        f.read(4)
        n    = int.from_bytes(f.read(4), 'big')
        rows = int.from_bytes(f.read(4), 'big')
        cols = int.from_bytes(f.read(4), 'big')
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(n, rows, cols)

def load_idx_labels(path):
    with open(path, 'rb') as f:
        f.read(4)
        n    = int.from_bytes(f.read(4), 'big')
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data

x_test = load_idx_images('t10k-images.idx3-ubyte')
y_test = load_idx_labels('t10k-labels.idx1-ubyte')

x_test_float = x_test.reshape(-1, 784).astype(np.float32) / 255.0
x_test_int   = (x_test_float * SCALE).astype(np.int32)

print(f'Test images: {x_test.shape}  |  Test labels: {y_test.shape}')

Test images: (10000, 28, 28)  |  Test labels: (10000,)


In [ ]:
# Use uint32 to match the HLS IP data width — same as the tutorial
input_buf  = allocate(shape=(TOTAL_IN,),  dtype=np.uint32)
output_buf = allocate(shape=(TOTAL_OUT,), dtype=np.uint32)

# Layout: [ x (784) | W (100352) | b (128) ]
# Copy W and b once — they don't change per inference
input_buf[INPUT_SIZE : INPUT_SIZE + W_SIZE] = W1_int.flatten().view(np.uint32)
input_buf[INPUT_SIZE + W_SIZE :]            = b1_int.flatten().view(np.uint32)

print('Buffers allocated and W, b loaded')

Buffers allocated and W, b loaded


In [ ]:
# Write 0x81 to set AP_START (bit 0) and AUTO_RESTART (bit 7)
# AUTO_RESTART means the kernel runs continuously without needing to be restarted
CONTROL_REGISTER = 0x0
hls_ip.write(CONTROL_REGISTER, 0x81)

print(hls_ip.register_map)

RegisterMap {
  CTRL = Register(AP_START=1, AP_DONE=0, AP_IDLE=0, AP_READY=0, RESERVED_1=0, AUTO_RESTART=1, RESERVED_2=0, INTERRUPT=0, RESERVED_3=0),
  GIER = Register(Enable=0, RESERVED=0),
  IP_IER = Register(CHAN0_INT_EN=0, CHAN1_INT_EN=0, RESERVED_0=0),
  IP_ISR = Register(CHAN0_INT_ST=0, CHAN1_INT_ST=0, RESERVED_0=0)
}


In [ ]:
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / np.sum(e)

def layer1_stream_hw(image_int32):
    input_buf[:INPUT_SIZE] = image_int32.view(np.uint32)

    output_buf_local = allocate(shape=(TOTAL_OUT,), dtype=np.uint32)

    dma_send.transfer(input_buf)
    dma_recv.transfer(output_buf_local)
    dma_send.wait()
    dma_recv.wait()

    result = np.frombuffer(output_buf_local, dtype=np.int32).copy()
    del output_buf_local
    return result

def forward_pass_hw(image_int32):
    a1_int   = layer1_stream_hw(image_int32)
    a1_float = a1_int.astype(np.float32) / (SCALE * SCALE)
    return softmax(a1_float @ W2 + b2)

print('Functions defined')

Functions defined


In [ ]:
img_int = x_test_int[0]
result  = layer1_stream_hw(img_int)

print(f'HW output (first 10): {result[:10]}')
print(f'Non-zero count: {np.count_nonzero(result)}')
print(f'Min: {result.min()}, Max: {result.max()}')

pred = forward_pass_hw(img_int)
print(f'Predicted: {np.argmax(pred)}, Actual: {y_test[0]}')

HW output (first 10): [199   0   0   0   0   0   0 426 138   0]
Non-zero count: 68
Min: 0, Max: 993
Predicted: 7, Actual: 7


In [ ]:
N = 100
img_int = x_test_int[0]

t0 = time.perf_counter()
for _ in range(N):
    layer1_stream_hw(img_int)
t1 = time.perf_counter()

hw_ms = (t1 - t0) * 1000 / N
print(f'HW Layer 1 only (avg over {N} runs): {hw_ms:.4f} ms per image')
print(f'SW baseline was 0.10 ms per image')

HW Layer 1 only (avg over 100 runs): 1.9785 ms per image
SW baseline was 0.10 ms per image


In [ ]:
import matplotlib.pyplot as plt

num_samples = 30
indices = np.random.choice(len(x_test_int), num_samples, replace=False)

t0 = time.perf_counter()
predictions = np.array([forward_pass_hw(x_test_int[i]) for i in indices])
t1 = time.perf_counter()

predicted_labels = np.argmax(predictions, axis=1)
total_ms     = (t1 - t0) * 1000
per_image_ms = total_ms / num_samples

fig, axes = plt.subplots(6, 5, figsize=(15, 6))
axes = axes.flatten()
for idx, ax in enumerate(axes):
    image = x_test[indices[idx]].reshape(28, 28)
    ax.imshow(image, cmap='gray')
    predicted = predicted_labels[idx]
    actual    = y_test[indices[idx]]
    color     = 'green' if predicted == actual else 'red'
    ax.set_title(f'Pred: {predicted}, Actual: {actual}', color=color, fontweight='bold')
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f'\nPrediction Summary:')
print(f'Correctly predicted: {np.sum(predicted_labels == y_test[indices])} / {num_samples}')
print(f'\nTiming (HW Layer 1 + SW Layer 2):')
print(f'  Total ({num_samples} images): {total_ms:.4f} ms')
print(f'  Per image: {per_image_ms:.4f} ms')

Prediction Summary:
Correctly predicted: 30 / 30

Timing (HW Layer 1 + SW Layer 2):
  Total (30 images): 67.0500 ms
  Per image: 2.2350 ms
